In [ ]:
import json

import numpy as np
from IPython.display import display
from matplotlib_inline.backend_inline import set_matplotlib_formats

from emu_renewal.constants import CONT_CMAP, DATA_PATH, FULL_RUN
from emu_renewal.outputs import get_policy_weight_summary
from emu_renewal.plotting import plot_manuscript_policy_weight_kdes
from emu_renewal.utils import (
    get_analysis_paths,
    get_countries_by_continent,
    get_country_name,
)

set_matplotlib_formats("svg")

In [ ]:
SEED = 42
N_PER_CONT = 3

all_countries = json.load(open(DATA_PATH / "config/oxcgrt_included.json"))
analysis_paths = get_analysis_paths(FULL_RUN, all_countries)

# Seed before prop_better so the supported pool is reproducible
np.random.seed(SEED)
summary = get_policy_weight_summary(analysis_paths)
supported = summary.index[summary["prop_better"] > 0.5].tolist()
countries_by_cont = get_countries_by_continent(supported)

rng = np.random.default_rng(SEED)
panel_countries = []
for cont in CONT_CMAP:
    if cont == "OC":
        continue
    pool = countries_by_cont[cont]
    chosen = rng.choice(pool, size=N_PER_CONT, replace=False)
    panel_countries.extend(chosen.tolist())

selection_df = summary.loc[panel_countries, ["prop_better", "strength"]].copy()
selection_df.insert(0, "country", [get_country_name(c) for c in panel_countries])

In [ ]:
fig = plot_manuscript_policy_weight_kdes(analysis_paths, panel_countries)
display(fig)